<h1>📘 Biofilter — Reports 101 (4.3.0)</h1>

How reports work now that they read a **bundle** instead of a database.

A bundle is a folder: parquet files plus a `manifest.json` that says what
is in them. It is immutable — the build that produced it is the version
of the data — and reports only ever read it.

This notebook covers the whole report API. Each individual report has
its own notebook in this folder.

## What changed from 4.2.0

| | 4.2.0 | 4.3.0 |
| --- | --- | --- |
| data | PostgreSQL, or a parquet bundle through an ORM bridge | the bundle, read natively |
| `bf.report.run()` returns | a DataFrame | a `ReportResult` |
| provenance | lost on export | written beside the file |

There is no relational path any more: a report reads a bundle or it does
not run.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

# A bundle is a directory. Point at the directory, not at its tables/.
BUNDLE = "/path/to/biofilter_data/bundles/20260914"

bf = Biofilter(bundle=BUNDLE, debug_mode=False)
bf

On the command line the same thing is `--bundle`:

```bash
biofilter --bundle /path/to/bundles/20260914 report list
```

Or put the path in `.biofilter.toml` once and drop the argument:

```toml
[database]
bundle = "./biofilter_data/bundles/20260914"
```

A relative path there is relative to the config file, not to where you
are, so it works from a notebook in a subdirectory. With that set,
`Biofilter()` and `biofilter report run ...` both find it, and
`biofilter config show` prints which bundle is in effect.

### 2. What reports exist

In [ ]:
import pandas as pd

reports = bf.report.list()
df = pd.DataFrame(reports)

pending = bf.report.pending_migration()
print(f"{len(df)} reports available; {len(pending)} still awaiting rewrite")
df[["name", "description"]]

Reports are being rewritten for the bundle one at a time.
`bf.report.pending_migration()` lists the ones that have not moved yet —
they live in `biofilter/modules/report_legacy/reports/` as reference for
whoever rewrites them, and **cannot be run**. Most could not run against
a 4.3.0 bundle anyway: they select columns the bundles stopped carrying.

```python
bf.report.pending_migration()[:5]
```

### 3. Ask a report about itself

In [ ]:
report_name = "annotate_gene"

print("columns:")
print(bf.report.available_columns(report_name))

print("\nexample input:")
print(bf.report.example_input(report_name))

In [ ]:
print(bf.report.explain(report_name))

### 4. Run one

In [ ]:
result = bf.report.run(report_name, input_data=["TP53", "BRCA1"])

# Native reports return a ReportResult, not a DataFrame.
print(type(result).__name__)
print(f"{result.num_rows} rows")

df = result.to_pandas()
df[["input_value", "gene_symbol", "hgnc_id", "chromosome", "status"]]

### 5. Provenance — which bundle produced this

`entity_id` and `variant_id` are **scoped to one bundle**. The same
integer means a different gene in the next build, and a stale id still
resolves — to the wrong row. Carrying the bundle id alongside the data is
what makes that detectable.

In [ ]:
result.provenance

### 6. Export

In [ ]:
# CSV, with a genes.csv.provenance.json written beside it.
written = result.write("genes.csv")
for path in written:
    print(path)

In [ ]:
# Parquet instead: the provenance travels inside the file's metadata,
# and list columns stay real lists rather than JSON strings.
result.write("genes.parquet")

### 7. Reading a result someone else produced

The sidecar is what lets you answer "where did this come from" months
later, and `build_record.json` in the bundle has the per-source detail
behind that id — which DTP, which version, which source URL.

In [ ]:
import json

with open("genes.csv.provenance.json") as fh:
    print(json.dumps(json.load(fh), indent=2))

---

## Working directly, without the facade

Reports are the packaged questions. For an ad-hoc one, open the bundle
and write SQL — it is the same engine the reports use.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(BUNDLE) as bundle:
    print(f"bundle {bundle.bundle_id}, {len(bundle.tables)} tables")

    # Every table is a view; query it as SQL and get Arrow back.
    out = bundle.con.execute("""
        SELECT g.name AS gene_group, count(*) AS genes
        FROM gene_masters gm
        JOIN gene_group_memberships m ON m.gene_id = gm.id
        JOIN gene_groups g ON g.id = m.group_id
        GROUP BY 1 ORDER BY genes DESC LIMIT 10
    """).to_arrow_table()

display(out.to_pandas())

### What is in this bundle

`bundle.tables` is the manifest, resolved: one entry per logical table,
with the files behind it. A partitioned table is many files and one view.

In [ ]:
with Bundle.open(BUNDLE) as bundle:
    inventory = pd.DataFrame(
        [
            {
                "table": t.name,
                "rows": t.rows,
                "files": len(t.files),
                "branch": t.branch,
            }
            for t in bundle.tables.values()
        ]
    ).sort_values("rows", ascending=False)

inventory.head(15)